In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
!pip -q install transformers datasets peft accelerate sentencepiece

In [18]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.8 MB/s eta 0:00:00:00:010:01


In [4]:

import torch
import pandas as pd
import numpy as np

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [5]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print(train.head())

   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that humans do not e...   
4  Simultaneity is rel

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train["label"] = train["answer"].map(label_map)

print(train.loc[150,"label"])

2


In [10]:
formatted = str(train.loc[0,"prompt"]) + " [SEP] " + str(train.loc[0,"B"])

print(formatted)

print(len(formatted))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
407


In [9]:
choices = []

for option in ["A","B","C","D","E"]:
    choices.append(
        str(train.loc[0,"prompt"]) + " [SEP] " + str(train.loc[0,option])
    )

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)

print(input_ids.shape)

torch.Size([1, 5, 128])


In [11]:
all_input_ids = []

for i in range(16):

    choices=[]

    for option in ["A","B","C","D","E"]:

        choices.append(
            str(train.loc[i,"prompt"])+" [SEP] "+str(train.loc[i,option])
        )

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    all_input_ids.append(enc["input_ids"])

all_input_ids=torch.stack(all_input_ids)

print(all_input_ids.shape)

print(all_input_ids.numel())

torch.Size([16, 5, 128])
10240


In [12]:
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

outputs = model(
    input_ids=input_ids,
    attention_mask=encoding["attention_mask"].unsqueeze(0)
)

print(outputs.logits.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])


In [13]:
label = torch.tensor([train.loc[0,"label"]])

outputs = model(
    input_ids=input_ids,
    attention_mask=encoding["attention_mask"].unsqueeze(0),
    labels=label
)

print(outputs.loss)

print(outputs.loss.ndim)

tensor(1.6182, grad_fn=<NllLossBackward0>)
0


In [19]:
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, config)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(trainable)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

295681


In [15]:
dataset = Dataset.from_pandas(train.iloc[:100])

def preprocess(example):

    first = [example["prompt"]]*5

    second = [
        example["A"],
        example["B"],
        example["C"],
        example["D"],
        example["E"]
    ]

    enc = tokenizer(
        first,
        second,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    enc["labels"] = label_map[example["answer"]]

    return enc

dataset = dataset.map(preprocess)

print(np.array(dataset[0]["input_ids"]).shape)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

(5, 128)


In [20]:
small = Dataset.from_pandas(train.iloc[:32])

small = small.map(preprocess)

small.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, config)

args = TrainingArguments(

    output_dir="./output",

    per_device_train_batch_size=4,

    gradient_accumulation_steps=1,

    max_steps=4,

    logging_steps=1,

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small
)

trainer.train()

print(trainer.state.global_step)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,1.725271
2,1.534214
3,1.603825
4,1.689587


4


In [17]:
example = train.iloc[0]

first = [example["prompt"]]*5

second = [
    example["A"],
    example["B"],
    example["C"],
    example["D"],
    example["E"]
]

enc = tokenizer(
    first,
    second,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

with torch.no_grad():

    out = model(
        input_ids=enc["input_ids"].unsqueeze(0),
        attention_mask=enc["attention_mask"].unsqueeze(0)
    )

prob = torch.softmax(out.logits, dim=1)

print(prob)

print(prob[0,4].item())

tensor([[0.1994, 0.1917, 0.1998, 0.2018, 0.2073]])
0.20731456577777863
